# 04 — Anytime evaluation and the cost meter (rel-f1)

**Purpose.** The compute results (RQ2, RQ4) rest on two instruments.

- **Anytime curves.** The next-window score at fixed optimiser steps of every
  update. Evaluating must not change what is trained.
- **Cost meter.** It splits each update's wall-clock into training compute, data
  loading, validation and anytime evaluation, and counts samples per source.
  Its buckets must add up, and the mixing shares it measures must match p.

All three checks run on the real rel-f1 model and graph.

**Gate.**
1. Final weights are bit-identical with and without anytime evaluation (CPU).
2. Every metered update's buckets account for its fit wall-clock within `RECONCILE_TOL`.
3. The measured new-data share of every mixing regime is within ±0.02 of p.

In [1]:
DATASET = "rel-f1"
TASK = "driver-dnf"
DEVICE = "auto"
SCALE = "smoke"
OUT = "notebooks/methodology/out"
THREADS = 8
HOLDOUT_FRAC = 0.2
STEPS = 120
RECONCILE_TOL = 0.10  # wall-clock not covered by any bucket (Lightning's own loop overhead)
MIX_MODES = {"replay_p50": 0.5, "replay_p25": 0.25}

In [2]:
# Parameters
SCALE = "smoke"
DEVICE = "cpu"


In [3]:
import json
import os
import shutil
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

import pandas as pd
import torch

from experiments.continuous_learning.chain_runner import ChainConfig, build_context, run_chain

torch.set_num_threads(THREADS)
if SCALE == "smoke":
    STEPS = min(STEPS, 60)
out = Path(OUT)
work = out / "04_work"
shutil.rmtree(work, ignore_errors=True)


def cfg(mode, variant, **kw):
    base = dict(device=DEVICE, torch_threads=THREADS, holdout_frac=HOLDOUT_FRAC, max_steps=STEPS,
                check_every=20, patiences=(1000,), max_updates=2)
    base.update(kw)
    return ChainConfig(DATASET, TASK, mode, 0, str(work), variant=variant, **base)


ctx = build_context(cfg("cumulative", "x"))

/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading Database object from /home/pelesjak/.cache/relbench/rel-f1/db...
Done in 0.05 seconds.


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.se

## 1. Anytime evaluation does not change training

The same two updates, once with the next window scored every 10 steps and once
with no anytime evaluation at all. Bitwise equality is only meaningful on one
CPU thread: multi-threaded CPU kernels (and GPU atomics) accumulate sums in a
varying order, so two identical runs drift apart even without the callback.

In [4]:
with_any = cfg("cumulative", "anytime_on", anytime_every=10, torch_threads=1)
without = cfg("cumulative", "anytime_off", anytime=False, torch_threads=1)
run_chain(with_any, ctx=ctx)
run_chain(without, ctx=ctx)

identical = {}
for e in (1, 2):
    a = torch.load(with_any.seed_dir() / f"ep{e:03d}" / "model.pt")
    b = torch.load(without.seed_dir() / f"ep{e:03d}" / "model.pt")
    identical[e] = all(torch.equal(a[k], b[k]) for k in a)
rec = json.loads((with_any.seed_dir() / "ep002" / "episode.json").read_text())
print("bit-identical final weights per update:", identical)
print("anytime points of update 2:", [r["step"] for r in rec["anytime"]][:8], "...")

/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


...

/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


bit-identical final weights per update: {1: True, 2: True}
anytime points of update 2: [0, 10, 20, 30, 40, 50, 60] ...


## 2. The cost meter's buckets add up

In [5]:
rows = []
for c in (with_any, without):
    for ep_dir in sorted(c.seed_dir().glob("ep*")):
        r = json.loads((ep_dir / "episode.json").read_text())
        s = r["cost"]["seconds"]
        accounted = s["train_compute"] + s["train_data"] + s["val"] + s.get("anytime", 0.0)
        rows.append({"variant": c.variant, "update": r["episode"], "steps": r["cost"]["steps"],
                     "fit wall s": round(r["cost"]["fit_wall_seconds"], 2),
                     "train compute s": round(s["train_compute"], 2), "data s": round(s["train_data"], 2),
                     "val s": round(s["val"], 2), "anytime s": round(s.get("anytime", 0.0), 2),
                     "unaccounted": round(1 - accounted / r["cost"]["fit_wall_seconds"], 3),
                     "val / train": round(s["val"] / max(s["train_compute"] + s["train_data"], 1e-9), 3)})
cost = pd.DataFrame(rows)
cost

,variant,update,steps,fit wall s,train compute s,data s,val s,anytime s,unaccounted,val / train
0,anytime_on,1,60,43.53,26.71,1.92,0.81,13.76,0.008,0.028
1,anytime_on,2,60,60.98,37.82,2.19,2.36,18.42,0.003,0.059
2,anytime_off,1,60,33.56,30.64,1.86,0.74,0.00,0.010,0.023
3,anytime_off,2,60,47.60,43.28,2.07,2.09,0.00,0.003,0.046


## 3. Mixing shares as measured during training

In [6]:
shares = []
for mode, p in MIX_MODES.items():
    c = cfg(mode, "mix", anytime=False)
    run_chain(c, ctx=ctx)
    for ep_dir in sorted(c.seed_dir().glob("ep*")):
        r = json.loads((ep_dir / "episode.json").read_text())
        by = r["cost"]["samples_by_source"]
        total = sum(by.values())
        has_old = r["train_rows"].get("old", 0) > 0
        shares.append({"mode": mode, "update": r["episode"], "p": p if has_old else 1.0,
                       "new share measured": round(by.get("new", 0) / total, 4), "samples": total,
                       "new rows": r["train_rows"]["new"], "old rows": r["train_rows"].get("old", 0)})
shares = pd.DataFrame(shares)
shares

/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


,mode,update,p,new share measured,samples,new rows,old rows
0,replay_p50,1,1.00,1.0000,7680,656,0
1,replay_p50,2,0.50,0.4883,7680,549,656
2,replay_p25,1,1.00,1.0000,7680,656,0
3,replay_p25,2,0.25,0.2471,7680,549,656


## Gate

In [7]:
assert all(identical.values()), identical
assert (cost["unaccounted"].abs() <= RECONCILE_TOL).all(), cost
assert ((shares["new share measured"] - shares["p"]).abs() <= 0.02).all(), shares
(out / f"04_anytime_cost_{DATASET}.json").write_text(json.dumps(
    {"identical": identical, "cost": cost.to_dict(orient="records"), "shares": shares.to_dict(orient="records")},
    indent=2, default=str))
print("GATE 04 PASSED")

GATE 04 PASSED
